# 1D CNN Gesture Classifier

## Overview

This notebook implements a **1D Convolutional Neural Network (CNN)** for dynamic hand-gesture recognition using data-glove time-series recordings.

### How 1D Convolution Works on Time-Series Data

In a 1D CNN the convolutional kernel slides along the **time dimension** while treating all sensor channels simultaneously as the feature dimension.  
Concretely, given an input tensor of shape `(batch, T, C)` where `T` is the number of time steps and `C` is the number of sensor channels, a `Conv1D` layer with kernel size `k` computes dot products over windows of `k` consecutive time steps across **all** `C` channels at once.  
This means:
- **Local temporal patterns** (e.g. a rapid finger extension) are captured by small kernels (k=3-5).
- **Multi-scale features** are learned by stacking blocks with different kernel sizes or by letting pooling progressively compress the time axis.
- **Translational invariance** along time is provided for free by the shared-weight convolution — the same wrist-flexion motion is detected regardless of where in the window it occurs.

### Why 1D CNN for Glove Data?

CNNs have been shown to be highly effective for multi-channel wearable sensor data.  
The seminal reference for this approach is:

> **Atzori et al. / Simeone et al.** — *"Data glove-based gesture recognition using CNN"* — uses 1D CNN applied to multi-channel sensor time series from a data glove, demonstrating that local temporal feature extraction via convolution outperforms hand-crafted feature pipelines on gesture classification benchmarks.  
> Also see: Ordóñez & Roggen (2016), *"Deep Convolutional and LSTM Recurrent Neural Networks for Multimodal Wearable Activity Recognition"*, Sensors — https://doi.org/10.3390/s16010115

Compared to the LSTM notebook (`01_LSTM.ipynb`), 1D CNNs:
- Train significantly **faster** (parallelisable over time)
- Are better at capturing **local** motifs (finger flick, joint snap)
- Require less data to converge for fixed-length windows

### Data Schema

Column naming convention: `{hand}_{segment}_{loc}_{channel}`  
e.g. `left_thumb_mid_ax`, `right_index_prox_pitch`, `left_thumb_flex_mcp`, `right_palm_az`

| Sensor group | Columns |
|---|---|
| Finger distal IMU (`mid`) | `ax, ay, az, pitch, roll, yaw` per finger × hand |
| Finger proximal IMU (`prox`) | `ax, ay, az, pitch, roll, yaw` per finger × hand |
| Flex sensors | `flex_mcp, flex_pip` per finger × hand |
| Palm IMU | `palm_ax/ay/az/pitch/roll/yaw` per hand |
| Wrist IMU | `wrist_ax/ay/az/pitch/roll/yaw` per hand |
| **EXCLUDED** | Quaternion columns `qw, qx, qy, qz` |

Sampling rate: **22 Hz** (as measured in the glove_visualiser notebook).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1 — Install / upgrade dependencies
# Run once; the kernel restart is NOT automatic, so re-run remaining cells.
# ─────────────────────────────────────────────────────────────────────────────
import sys, subprocess

pkgs = [
    "tensorflow",
    "numpy",
    "pandas",
    "scikit-learn",
    "scipy",
    "matplotlib",
    "seaborn",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade"] + pkgs,
    check=True,
)
print("Dependencies ready.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2 — Imports
# ─────────────────────────────────────────────────────────────────────────────
import os
import sys
import json
import warnings
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")

# Add utils to path so we can import data_loader from sibling package
NOTEBOOK_DIR = Path(os.path.abspath(""))
sys.path.insert(0, str(NOTEBOOK_DIR))

from utils.data_loader import build_dataset, split_dataset, build_column_groups

print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")
print(f"Notebook directory : {NOTEBOOK_DIR}")

# Reproducibility seed for NumPy / TF random ops
GLOBAL_SEED = 42
tf.random.set_seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3 — CONFIG  ← edit everything here; downstream cells read these names
# ─────────────────────────────────────────────────────────────────────────────

# ── DATA ──────────────────────────────────────────────────────────────────────
DATA_ROOT = '../data'
FS_HZ     = 22.0          # glove sampling rate (Hz)

# ── COLUMN SELECTION ──────────────────────────────────────────────────────────
USE_LEFT_HAND     = True
USE_RIGHT_HAND    = True
USE_DISTAL_IMU    = True   # distal phalanx IMUs  (mid)
USE_PROXIMAL_IMU  = True   # proximal phalanx IMUs (prox)
USE_ACCELEROMETER = True   # ax, ay, az channels
USE_ORIENTATION   = True   # pitch, roll, yaw channels
USE_FLEX_SENSORS  = True   # flex_mcp, flex_pip per finger
USE_PALM_IMU      = True   # palm IMU
USE_WRIST_IMU     = True   # wrist IMU

# ── PREPROCESSING ─────────────────────────────────────────────────────────────
FILTER_TYPE     = 'butterworth_lp'  # see utils/data_loader.apply_filter
TARGET_LEN      = 110               # all samples resampled to this length
NORMALIZATION   = 'minmax'          # 'minmax' | 'zscore' | 'none'
PER_SAMPLE_NORM = True              # normalise each sample independently

# ── DATASET SPLIT ─────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.10
RANDOM_SEED = 42

# ── MODEL HYPERPARAMETERS ─────────────────────────────────────────────────────
# 1D CNN architecture (applied along the time axis)
# Each entry: (filters, kernel_size)
CONV_BLOCKS = [
    (64,  5),   # Block 1 – broad temporal context, 64 feature maps
    (128, 3),   # Block 2 – finer detail, 128 feature maps
    (256, 3),   # Block 3 – high-level abstractions, 256 feature maps
]
USE_BATCH_NORM = True
USE_MAX_POOL   = True       # MaxPooling1D after each block
POOL_SIZE      = 2          # halves the time dimension per block
DROPOUT_RATE   = 0.4
DENSE_UNITS    = [128, 64]  # FC layers before the softmax output head
GLOBAL_POOL    = 'average'  # 'average' | 'max' — collapses remaining time dim

# ── TRAINING ──────────────────────────────────────────────────────────────────
EPOCHS              = 60
BATCH_SIZE          = 32
LEARNING_RATE       = 1e-3
EARLY_STOP_PATIENCE = 10

# ── OUTPUT PATHS ──────────────────────────────────────────────────────────────
MODEL_SAVE_PATH   = 'saved_models/02_cnn_gesture.keras'
RESULTS_SAVE_PATH = 'results/02_cnn_results.json'

print("Configuration loaded.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4 — Build feature column list from CONFIG flags
# ─────────────────────────────────────────────────────────────────────────────

def build_feature_cols(
    use_left=True, use_right=True,
    use_distal=True, use_proximal=True,
    use_accel=True, use_orient=True,
    use_flex=True, use_palm=True, use_wrist=True,
):
    """
    Construct the desired feature column list from boolean flags.

    The naming convention is {hand}_{finger}_{loc}_{channel}:
      - loc  ∈ {'mid', 'prox'}      (per-finger IMUs)
      - channel ∈ {'ax','ay','az'}  (accelerometer) or {'pitch','roll','yaw'} (orientation)
    Flex sensors are named {hand}_{finger}_flex_{mcp|pip}.
    Palm/wrist are {hand}_palm_{ch} / {hand}_wrist_{ch}.
    """
    from utils.data_loader import FINGERS, HANDS, LOCS, IMU_CHANNELS, FLEX_CHANNELS

    ACCEL_CH  = ["ax", "ay", "az"]
    ORIENT_CH = ["pitch", "roll", "yaw"]

    # Determine which channels to include from each IMU
    imu_channels = []
    if use_accel:  imu_channels += ACCEL_CH
    if use_orient: imu_channels += ORIENT_CH

    # Determine which hands to include
    hands = []
    if use_left:  hands.append("left")
    if use_right: hands.append("right")

    # Determine which IMU locations (mid/prox) to include
    locs = []
    if use_distal:   locs.append("mid")
    if use_proximal: locs.append("prox")

    cols = []

    for hand in hands:
        for finger in FINGERS:
            # Per-finger IMU columns
            if imu_channels:
                for loc in locs:
                    for ch in imu_channels:
                        cols.append(f"{hand}_{finger}_{loc}_{ch}")
            # Flex sensor columns
            if use_flex:
                for ch in FLEX_CHANNELS:  # ['flex_mcp', 'flex_pip']
                    cols.append(f"{hand}_{finger}_{ch}")

        # Palm IMU
        if use_palm and imu_channels:
            for ch in imu_channels:
                cols.append(f"{hand}_palm_{ch}")

        # Wrist IMU
        if use_wrist and imu_channels:
            for ch in imu_channels:
                cols.append(f"{hand}_wrist_{ch}")

    # De-duplicate while preserving order
    seen = set()
    unique_cols = []
    for c in cols:
        if c not in seen:
            seen.add(c)
            unique_cols.append(c)

    return unique_cols


FEATURE_COLS = build_feature_cols(
    use_left=USE_LEFT_HAND,
    use_right=USE_RIGHT_HAND,
    use_distal=USE_DISTAL_IMU,
    use_proximal=USE_PROXIMAL_IMU,
    use_accel=USE_ACCELEROMETER,
    use_orient=USE_ORIENTATION,
    use_flex=USE_FLEX_SENSORS,
    use_palm=USE_PALM_IMU,
    use_wrist=USE_WRIST_IMU,
)

print(f"Requested feature columns : {len(FEATURE_COLS)}")
print("First 10:", FEATURE_COLS[:10])
print("Last  10:", FEATURE_COLS[-10:])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5 — Load dataset and show class distribution
# ─────────────────────────────────────────────────────────────────────────────

# build_dataset will gracefully skip columns that don't exist in the CSVs,
# so passing the full FEATURE_COLS list is safe even if some glove channels
# are absent in a particular data-collection session.
X, y, CLASS_NAMES, FEATURE_COLS_USED = build_dataset(
    data_root       = DATA_ROOT,
    feature_cols    = FEATURE_COLS,
    filter_type     = FILTER_TYPE,
    fs              = FS_HZ,
    target_len      = TARGET_LEN,
    normalization   = NORMALIZATION,
    per_sample_norm = PER_SAMPLE_NORM,
    exclude_quat    = True,
)

N_CLASSES  = len(CLASS_NAMES)
N_FEATURES = X.shape[2]

print(f"\nDataset shape  : X={X.shape}  y={y.shape}")
print(f"Classes ({N_CLASSES})    : {CLASS_NAMES}")
print(f"Features used  : {N_FEATURES} (of {len(FEATURE_COLS)} requested)")
if len(FEATURE_COLS_USED) < len(FEATURE_COLS):
    missing = set(FEATURE_COLS) - set(FEATURE_COLS_USED)
    print(f"  ⚠ {len(missing)} columns absent in data — ignored.")

# ── Class distribution bar chart ─────────────────────────────────────────────
counts = np.bincount(y, minlength=N_CLASSES)

fig, ax = plt.subplots(figsize=(max(6, N_CLASSES * 0.9), 4))
bars = ax.bar(CLASS_NAMES, counts, color='#20808D', edgecolor='white', linewidth=0.8)
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_title('Class distribution in loaded dataset', fontsize=13, fontweight='bold')
ax.set_xlabel('Gesture class')
ax.set_ylabel('Sample count')
ax.set_xticklabels(CLASS_NAMES, rotation=35, ha='right', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

# ── Quick data sanity checks ──────────────────────────────────────────────────
print(f"\nX min={X.min():.4f}  max={X.max():.4f}  NaNs={np.isnan(X).sum()}")
print(f"Label distribution: {dict(zip(CLASS_NAMES, counts))}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6 — Stratified train / val / test split
# ─────────────────────────────────────────────────────────────────────────────

(X_train, y_train), (X_val, y_val), (X_test, y_test) = split_dataset(
    X, y,
    train_ratio = TRAIN_RATIO,
    val_ratio   = VAL_RATIO,
    seed        = RANDOM_SEED,
)

# Convert labels to one-hot for categorical_crossentropy
y_train_oh = tf.keras.utils.to_categorical(y_train, N_CLASSES)
y_val_oh   = tf.keras.utils.to_categorical(y_val,   N_CLASSES)
y_test_oh  = tf.keras.utils.to_categorical(y_test,  N_CLASSES)

print("\nShapes after split:")
print(f"  X_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"  X_val   : {X_val.shape}   y_val   : {y_val.shape}")
print(f"  X_test  : {X_test.shape}   y_test  : {y_test.shape}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 7 — Build the 1D CNN model
# ─────────────────────────────────────────────────────────────────────────────

def build_cnn_model(
    seq_len,
    n_features,
    n_classes,
    conv_blocks,
    use_batch_norm=True,
    use_max_pool=True,
    pool_size=2,
    dropout_rate=0.4,
    dense_units=None,
    global_pool='average',
):
    """
    Build a configurable 1D CNN for sequence classification.

    Architecture explanation
    ------------------------
    Input shape: (batch, seq_len, n_features)

    Each Conv1D block does the following:
      1. Conv1D(filters, kernel_size, padding='causal')
         - 'causal' padding means the convolution only looks at the current
           and past time steps — this mirrors how the model would run in
           real-time inference, though for offline classification 'same'
           padding is also fine (change below if desired).
      2. Optional BatchNormalization — stabilises activations, allows higher LR
      3. ReLU activation
      4. Optional MaxPooling1D — halves the temporal resolution, doubles the
         effective receptive field of subsequent layers
      5. Dropout — regularisation against overfitting

    After all conv blocks:
      - GlobalAveragePooling1D *or* GlobalMaxPooling1D collapses the remaining
        time dimension into a fixed-size feature vector regardless of seq_len.
      - Dense FC layers for classification.
      - Softmax output head.

    Receptive Field
    ---------------
    The receptive field R grows with each block:
      R_0 = 1 (before any convolution)
      After Conv1D(k) with stride 1: R += (k-1) * stride_so_far
      After MaxPool(p): stride_so_far *= p
    We compute and print this below.

    Parameters
    ----------
    seq_len      : int — number of time steps (TARGET_LEN)
    n_features   : int — number of sensor channels (C)
    n_classes    : int — number of gesture classes
    conv_blocks  : list of (filters, kernel_size) tuples
    use_batch_norm : bool
    use_max_pool : bool — MaxPooling1D after each block
    pool_size    : int  — pooling window size
    dropout_rate : float
    dense_units  : list[int] — number of units in each FC layer
    global_pool  : 'average' | 'max'

    Returns
    -------
    model : tf.keras.Model
    """
    if dense_units is None:
        dense_units = [128]

    inputs = keras.Input(shape=(seq_len, n_features), name='sensor_input')
    x = inputs

    # ── Convolutional blocks ───────────────────────────────────────────────────
    for block_idx, (filters, kernel_size) in enumerate(conv_blocks, start=1):
        # Conv1D applies a 1-D kernel of length `kernel_size` across the TIME
        # axis.  All `n_features` input channels are combined in each kernel
        # (similar to how 2D conv combines RGB channels).
        x = layers.Conv1D(
            filters=filters,
            kernel_size=kernel_size,
            padding='same',    # preserve temporal length; switch to 'causal'
                               # for online / streaming inference
            use_bias=not use_batch_norm,  # bias is redundant when BN follows
            name=f'conv_block{block_idx}_conv',
        )(x)

        if use_batch_norm:
            # Batch normalisation normalises the activations across the batch
            # and spatial (time) dimensions — makes training faster and less
            # sensitive to weight initialisation.
            x = layers.BatchNormalization(name=f'conv_block{block_idx}_bn')(x)

        x = layers.Activation('relu', name=f'conv_block{block_idx}_relu')(x)

        if use_max_pool:
            # MaxPooling1D selects the maximum activation in each pool window.
            # This provides translational invariance over short temporal shifts
            # and progressively compresses the sequence length.
            x = layers.MaxPooling1D(
                pool_size=pool_size,
                name=f'conv_block{block_idx}_pool',
            )(x)

        x = layers.Dropout(dropout_rate, name=f'conv_block{block_idx}_drop')(x)

    # ── Global pooling — collapse the remaining time dimension ─────────────────
    # After the conv+pool stack the sequence length may still be >1 (e.g. 13
    # time steps for TARGET_LEN=110, 3 MaxPool(2) blocks, and 'same' conv).
    # GlobalAveragePooling averages ALL remaining time steps (soft attention
    # analogue), whereas GlobalMaxPooling picks the strongest activation.
    if global_pool == 'average':
        x = layers.GlobalAveragePooling1D(name='global_avg_pool')(x)
    elif global_pool == 'max':
        x = layers.GlobalMaxPooling1D(name='global_max_pool')(x)
    else:
        raise ValueError(f"global_pool must be 'average' or 'max', got '{global_pool}'")

    # ── Fully connected classification head ───────────────────────────────────
    for fc_idx, units in enumerate(dense_units, start=1):
        x = layers.Dense(units, use_bias=not use_batch_norm,
                          name=f'fc{fc_idx}')(x)
        if use_batch_norm:
            x = layers.BatchNormalization(name=f'fc{fc_idx}_bn')(x)
        x = layers.Activation('relu', name=f'fc{fc_idx}_relu')(x)
        x = layers.Dropout(dropout_rate, name=f'fc{fc_idx}_drop')(x)

    outputs = layers.Dense(n_classes, activation='softmax', name='output')(x)

    model = Model(inputs=inputs, outputs=outputs, name='1D_CNN_Gesture')
    return model


# ── Instantiate ───────────────────────────────────────────────────────────────
model = build_cnn_model(
    seq_len       = TARGET_LEN,
    n_features    = N_FEATURES,
    n_classes     = N_CLASSES,
    conv_blocks   = CONV_BLOCKS,
    use_batch_norm= USE_BATCH_NORM,
    use_max_pool  = USE_MAX_POOL,
    pool_size     = POOL_SIZE,
    dropout_rate  = DROPOUT_RATE,
    dense_units   = DENSE_UNITS,
    global_pool   = GLOBAL_POOL,
)

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy'],
)

model.summary()

# ── Receptive field computation ───────────────────────────────────────────────
print("\n" + "─" * 55)
print("Receptive Field Analysis")
print("─" * 55)
rf = 1
stride_acc = 1  # accumulated stride from pooling layers
for block_idx, (_, k) in enumerate(CONV_BLOCKS, start=1):
    rf += (k - 1) * stride_acc
    print(f"  After Conv Block {block_idx} (kernel={k}): RF = {rf} time steps")
    if USE_MAX_POOL:
        stride_acc *= POOL_SIZE
        print(f"  After MaxPool ({POOL_SIZE}):         stride_acc = {stride_acc}")

print(f"\nFinal receptive field : {rf} time steps")
print(f"Total sequence length  : {TARGET_LEN} time steps")
print(f"At {FS_HZ} Hz, RF spans : {rf / FS_HZ * 1000:.1f} ms")
print("─" * 55)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8 — Training with callbacks + learning-curve plots
# ─────────────────────────────────────────────────────────────────────────────

# Create output directories if they don't exist
Path('saved_models').mkdir(exist_ok=True)
Path('results').mkdir(exist_ok=True)

# ── Callbacks ─────────────────────────────────────────────────────────────────
cb_list = [
    # Stop training when val_loss hasn't improved for EARLY_STOP_PATIENCE epochs.
    # restore_best_weights=True automatically reloads the best checkpoint.
    callbacks.EarlyStopping(
        monitor              = 'val_loss',
        patience             = EARLY_STOP_PATIENCE,
        restore_best_weights = True,
        verbose              = 1,
    ),
    # Reduce learning rate on plateau — helps escape flat regions
    callbacks.ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,
        patience = max(3, EARLY_STOP_PATIENCE // 3),
        min_lr   = 1e-6,
        verbose  = 1,
    ),
    # Save best model checkpoint to disk
    callbacks.ModelCheckpoint(
        filepath         = MODEL_SAVE_PATH,
        monitor          = 'val_accuracy',
        save_best_only   = True,
        save_weights_only= False,
        verbose          = 0,
    ),
]

print(f"Training for up to {EPOCHS} epochs  "
      f"(early stop patience={EARLY_STOP_PATIENCE})...")

history = model.fit(
    X_train, y_train_oh,
    validation_data = (X_val, y_val_oh),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    callbacks       = cb_list,
    verbose         = 1,
)

# ── Learning curves ───────────────────────────────────────────────────────────
hist = history.history
epochs_ran = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# — Loss —
axes[0].plot(epochs_ran, hist['loss'],     label='Train loss',
             color='#20808D', linewidth=2)
axes[0].plot(epochs_ran, hist['val_loss'], label='Val loss',
             color='#A84B2F', linewidth=2, linestyle='--')
best_epoch = np.argmin(hist['val_loss']) + 1
axes[0].axvline(best_epoch, color='gray', linestyle=':', alpha=0.7,
                label=f'Best epoch ({best_epoch})')
axes[0].set_title('Cross-entropy loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=9)
axes[0].spines[['top', 'right']].set_visible(False)

# — Accuracy —
axes[1].plot(epochs_ran, hist['accuracy'],     label='Train acc',
             color='#20808D', linewidth=2)
axes[1].plot(epochs_ran, hist['val_accuracy'], label='Val acc',
             color='#A84B2F', linewidth=2, linestyle='--')
axes[1].axvline(best_epoch, color='gray', linestyle=':', alpha=0.7,
                label=f'Best epoch ({best_epoch})')
axes[1].set_title('Classification accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.02)
axes[1].legend(fontsize=9)
axes[1].spines[['top', 'right']].set_visible(False)

fig.suptitle('1D CNN — Training curves', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('results/02_cnn_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

best_val_acc = max(hist['val_accuracy'])
print(f"\nBest validation accuracy : {best_val_acc:.4f} (epoch {best_epoch})")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 9 — Evaluation: classification report + confusion matrix heatmap
# ─────────────────────────────────────────────────────────────────────────────

# Load best checkpoint (EarlyStopping already restored weights, but
# explicitly loading the saved .keras file guarantees reproducibility)
try:
    best_model = keras.models.load_model(MODEL_SAVE_PATH)
    print(f"Loaded best model from: {MODEL_SAVE_PATH}")
except Exception as e:
    print(f"Could not load checkpoint ({e}), using current in-memory model.")
    best_model = model

# ── Test-set predictions ──────────────────────────────────────────────────────
y_pred_prob = best_model.predict(X_test, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)

test_loss, test_acc = best_model.evaluate(X_test, y_test_oh, verbose=0)
print(f"\nTest loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_acc:.4f}")

# ── Classification report ─────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("Classification report (test set)")
print("─" * 60)
print(classification_report(
    y_test, y_pred,
    target_names = CLASS_NAMES,
    digits       = 4,
))

# ── Confusion matrix heatmap ──────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)  # row-normalised

fig, axes = plt.subplots(1, 2, figsize=(max(10, N_CLASSES * 1.3),
                                         max(8,  N_CLASSES * 1.0)))

for ax, data, fmt, title in zip(
    axes,
    [cm,      cm_norm],
    ['d',     '.2f'],
    ['Raw count confusion matrix', 'Normalised confusion matrix (row %)'],
):
    sns.heatmap(
        data,
        annot          = True,
        fmt            = fmt,
        cmap           = 'Blues',
        xticklabels    = CLASS_NAMES,
        yticklabels    = CLASS_NAMES,
        linewidths     = 0.4,
        linecolor      = '#cccccc',
        ax             = ax,
        annot_kws      = {'size': max(6, 10 - N_CLASSES // 3)},
    )
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

fig.suptitle('1D CNN — Confusion matrices (test set)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('results/02_cnn_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 10 — Save model and results JSON
# ─────────────────────────────────────────────────────────────────────────────

from sklearn.metrics import classification_report as sk_cr

# Build a serialisable results dict
cr_dict = sk_cr(
    y_test, y_pred,
    target_names = CLASS_NAMES,
    output_dict  = True,
)

results = {
    "model"         : "1D_CNN_Gesture",
    "timestamp"     : datetime.datetime.now().isoformat(),

    # Config snapshot
    "config": {
        "filter_type"       : FILTER_TYPE,
        "target_len"        : TARGET_LEN,
        "normalization"     : NORMALIZATION,
        "per_sample_norm"   : PER_SAMPLE_NORM,
        "conv_blocks"       : CONV_BLOCKS,
        "use_batch_norm"    : USE_BATCH_NORM,
        "use_max_pool"      : USE_MAX_POOL,
        "pool_size"         : POOL_SIZE,
        "dropout_rate"      : DROPOUT_RATE,
        "dense_units"       : DENSE_UNITS,
        "global_pool"       : GLOBAL_POOL,
        "epochs"            : EPOCHS,
        "batch_size"        : BATCH_SIZE,
        "learning_rate"     : LEARNING_RATE,
        "early_stop_patience": EARLY_STOP_PATIENCE,
    },

    # Dataset info
    "dataset": {
        "n_samples"     : int(X.shape[0]),
        "n_timesteps"   : int(X.shape[1]),
        "n_features"    : int(N_FEATURES),
        "n_classes"     : int(N_CLASSES),
        "class_names"   : CLASS_NAMES,
        "feature_cols"  : FEATURE_COLS_USED,
        "train_size"    : int(X_train.shape[0]),
        "val_size"      : int(X_val.shape[0]),
        "test_size"     : int(X_test.shape[0]),
    },

    # Training outcomes
    "training": {
        "epochs_ran"         : len(hist['loss']),
        "best_epoch"         : int(best_epoch),
        "best_val_accuracy"  : float(best_val_acc),
        "best_val_loss"      : float(min(hist['val_loss'])),
        "final_train_accuracy": float(hist['accuracy'][-1]),
        "final_train_loss"   : float(hist['loss'][-1]),
    },

    # Test-set performance
    "evaluation": {
        "test_loss"          : float(test_loss),
        "test_accuracy"      : float(test_acc),
        "classification_report": cr_dict,
        "confusion_matrix"   : cm.tolist(),
    },
}

# Save JSON
with open(RESULTS_SAVE_PATH, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Results saved → {RESULTS_SAVE_PATH}")

# Save the best model (already saved by ModelCheckpoint, but save again for safety)
best_model.save(MODEL_SAVE_PATH)
print(f"Model saved  → {MODEL_SAVE_PATH}")

# Pretty-print key metrics
print("\n" + "═" * 50)
print("FINAL RESULTS — 1D CNN")
print("═" * 50)
print(f"  Test accuracy   : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  Test loss       : {test_loss:.4f}")
print(f"  Macro F1        : {cr_dict['macro avg']['f1-score']:.4f}")
print(f"  Weighted F1     : {cr_dict['weighted avg']['f1-score']:.4f}")
print("═" * 50)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 11 — Feature importance via gradient saliency (optional)
#
# Gradient saliency shows WHICH TIME STEPS and WHICH CHANNELS the network
# attended to most when making its prediction for a given sample.
#
# Method:
#   1. Pick one test sample per class.
#   2. Use tf.GradientTape to compute d(predicted_score) / d(input).
#   3. Take the absolute value and average over the channel dimension to get
#      temporal saliency, or average over the time dimension to get per-channel
#      saliency.
#   4. Display as colour-coded heatmaps.
#
# This is a first-order approximation.  For richer attribution consider
# Integrated Gradients or SHAP DeepExplainer.
# ─────────────────────────────────────────────────────────────────────────────

# ── Pick one representative sample per class ──────────────────────────────────
SALIENCY_MAX_CLASSES = min(N_CLASSES, 6)  # limit to 6 panels for readability

sample_indices = []
sample_labels  = []
for cls in range(SALIENCY_MAX_CLASSES):
    idx_candidates = np.where(y_test == cls)[0]
    if len(idx_candidates) == 0:
        continue
    # Prefer correctly classified samples for cleaner saliency
    correct = idx_candidates[y_pred[idx_candidates] == cls]
    chosen  = correct[0] if len(correct) > 0 else idx_candidates[0]
    sample_indices.append(chosen)
    sample_labels.append(cls)

# ── Compute gradients via GradientTape ───────────────────────────────────────
def compute_saliency(model, X_sample, true_class):
    """
    Compute input-gradient saliency map for one sample.

    Returns
    -------
    saliency : np.ndarray shape (T, C) — absolute gradient magnitude
    """
    x_tensor = tf.constant(X_sample[np.newaxis, :, :], dtype=tf.float32)  # (1, T, C)

    with tf.GradientTape() as tape:
        tape.watch(x_tensor)
        preds = model(x_tensor, training=False)          # (1, n_classes)
        score = preds[0, true_class]                     # scalar — target class score

    grads    = tape.gradient(score, x_tensor)            # (1, T, C)
    saliency = tf.abs(grads[0]).numpy()                  # (T, C)
    return saliency


saliency_maps = []
for idx, cls in zip(sample_indices, sample_labels):
    sal = compute_saliency(best_model, X_test[idx], cls)
    saliency_maps.append(sal)

print(f"Computed saliency maps for {len(saliency_maps)} classes.")

# ── Plot 1: Temporal saliency (averaged across channels) ──────────────────────
n_panels = len(saliency_maps)
ncols    = min(n_panels, 3)
nrows    = (n_panels + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(ncols * 5, nrows * 2.5),
                          squeeze=False)
time_axis = np.arange(TARGET_LEN) / FS_HZ * 1000  # ms

for panel_idx, (sal, cls) in enumerate(zip(saliency_maps, sample_labels)):
    ax = axes[panel_idx // ncols][panel_idx % ncols]

    # Average absolute gradient over channel dimension → temporal profile
    temporal_sal = sal.mean(axis=1)  # (T,)
    temporal_sal = (temporal_sal - temporal_sal.min()) / \
                   (temporal_sal.max() - temporal_sal.min() + 1e-9)  # normalise to [0,1]

    ax.fill_between(time_axis, temporal_sal, alpha=0.4, color='#20808D')
    ax.plot(time_axis, temporal_sal, color='#1B474D', linewidth=1.2)

    # Highlight top-3 most salient windows
    top_t = np.argsort(temporal_sal)[-3:]
    ax.scatter(time_axis[top_t], temporal_sal[top_t],
               color='#A84B2F', s=40, zorder=5)

    # Was this sample correctly classified?
    pred_cls   = y_pred[sample_indices[panel_idx]]
    correct_str = '✓' if pred_cls == cls else f'✗→{CLASS_NAMES[pred_cls]}'

    ax.set_title(f"{CLASS_NAMES[cls]} {correct_str}", fontsize=10, fontweight='bold')
    ax.set_xlabel('Time (ms)', fontsize=8)
    ax.set_ylabel('Norm. saliency', fontsize=8)
    ax.set_ylim(-0.05, 1.1)
    ax.spines[['top', 'right']].set_visible(False)

# Hide unused panels
for extra in range(n_panels, nrows * ncols):
    axes[extra // ncols][extra % ncols].set_visible(False)

fig.suptitle('1D CNN — Temporal saliency (gradient magnitude averaged over channels)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/02_cnn_temporal_saliency.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Plot 2: Channel saliency heatmap (time × channel) for class 0 ─────────────
# Show the full 2D saliency map for the first sample as a heat map.
# This reveals which specific (time step, sensor channel) combinations
# were most informative for the network's decision.
if len(saliency_maps) > 0:
    sal0     = saliency_maps[0]   # (T, C)
    cls0     = sample_labels[0]
    n_show_c = min(N_FEATURES, 30)  # cap at 30 channels for legibility

    # Pick the n_show_c channels with the highest mean saliency
    channel_mean_sal = sal0.mean(axis=0)  # (C,)
    top_channel_idx  = np.argsort(channel_mean_sal)[::-1][:n_show_c]
    top_channel_names = [FEATURE_COLS_USED[i] for i in top_channel_idx]

    sal_subset = sal0[:, top_channel_idx].T  # (n_show_c, T)

    fig, ax = plt.subplots(figsize=(12, max(4, n_show_c * 0.35)))
    im = ax.imshow(
        sal_subset,
        aspect     = 'auto',
        origin     = 'upper',
        cmap       = 'YlOrRd',
        extent     = [0, TARGET_LEN / FS_HZ * 1000, n_show_c, 0],
    )
    ax.set_yticks(np.arange(n_show_c) + 0.5)
    ax.set_yticklabels(top_channel_names, fontsize=7)
    ax.set_xlabel('Time (ms)', fontsize=10)
    ax.set_title(
        f'2D saliency map — class: {CLASS_NAMES[cls0]} '
        f'(top {n_show_c} channels by mean gradient)',
        fontsize=11, fontweight='bold',
    )
    cbar = fig.colorbar(im, ax=ax, pad=0.01)
    cbar.set_label('|∂score/∂input|', fontsize=9)
    plt.tight_layout()
    plt.savefig('results/02_cnn_channel_saliency.png', dpi=150, bbox_inches='tight')
    plt.show()

print("Saliency analysis complete.  Figures saved in results/.")